# Dev workflow: vectorized calc + full backtest sanity check

Two things this notebook exists to verify, end to end, against a real
fetched price panel (no synthetic data):

1. **The vectorized calculation is right** -- `src2/vector_calc`'s
   matrix-wide scorer/filter/rank pipeline (`price -> metrics ->
   filter/rank -> target_weight`) is cross-checked cell-by-cell against
   the pre-existing per-ticker `src_old/scorers/mean_reversion_scorers.MeanReversionScorer`
   class, to confirm the vectorized version agrees with the original
   before trusting it.
2. **`iteration.run_daily_iteration` runs successfully** -- the precomputed
   target_weight table feeds cleanly into the stateful day-by-day
   execution loop and produces a sane NAV curve.

`src` was renamed to `src_old`; this notebook now runs entirely on `src2`
except for the one deliberate cross-check import of `src_old`'s original
scorer class in section 5.


In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd

from src2 import vector_calc, data_ingest, iteration, evaluation
from src_old.scorers.mean_reversion_scorers import MeanReversionScorer


## 1. Synthetic price panel (date x ticker)

In [2]:
tickers='ALGN,AAPL,PANW,AMD,ZS,MDLZ,XEL,VRSN,INSM,ON,WDAY,ENPH,DLTR,ADI,COST,ABNB,CEG,MSFT,GOOGL,CDNS,PYPL,FAST,JD,MTCH,SNPS,LULU,CMCSA,ADBE,ADSK'.split(',')

In [3]:
broker = data_ingest.DataBroker(tickers=tickers, start_date="2022-01-01", end_date="2026-04-01")
universe_data = broker.fetch_universe_data()


[                       0%                       ]

[***                    7%                       ]  2 of 29 completed

[*****                 10%                       ]  3 of 29 completed

[**********            21%                       ]  6 of 29 completed

[*************         28%                       ]  8 of 29 completed

...

[**********************76%***********            ]  22 of 29 completed

[**********************79%*************          ]  23 of 29 completed

[**********************83%***************        ]  24 of 29 completed

[**********************90%******************     ]  26 of 29 completed

[*********************100%***********************]  29 of 29 completed

In [4]:
price_df=universe_data['price']

In [5]:
# rng = np.random.default_rng(0)
# dates = pd.bdate_range("2022-01-01", "2023-12-31")
# tickers = [f"T{i}" for i in range(8)]

# log_returns = rng.normal(loc=0.0002, scale=0.015, size=(len(dates), len(tickers)))
# price_df = pd.DataFrame(100 * np.exp(np.cumsum(log_returns, axis=0)), index=dates, columns=tickers)
# price_df.head()

## 2. Rebalance dates (month-end, matches iteration.generate_rebalance_dates' default)


In [6]:
world_data_dict=universe_data

In [7]:
rebalance_dates = iteration.generate_rebalance_dates(world_data_dict["price"], interval="ME")


In [8]:
rebalance_dates[:3]

[Timestamp('2022-01-31 00:00:00'),
 Timestamp('2022-02-28 00:00:00'),
 Timestamp('2022-03-31 00:00:00')]

In [9]:
# rebalance_dates = pd.Series(dates, index=dates).resample("ME").last().dropna().tolist()
# len(rebalance_dates), rebalance_dates[:3]

## 3. Metrics table -- one row per (date, ticker), fully inspectable

In [10]:
WINDOW = 60

df_metrics, df_ranked = vector_calc.build_target_weight_table(
    price_df,
    rebalance_dates,
    score_fn=vector_calc.rolling_mean_reversion_score,
    window=WINDOW,
    top_percent=0.25,
    allocation_type="equal",
)
df_metrics.head(10)


,date,ticker,score,z_score,raw_z_score,window_mean,window_std
0,2022-01-31,ALGN,0.466693,-0.466693,-0.466693,520.869496,55.517293
1,2022-01-31,AAPL,-0.727530,0.727530,0.727530,166.062898,6.609396
2,2022-01-31,PANW,-0.398864,0.398864,0.398864,84.915417,3.304162
3,2022-01-31,AMD,0.943130,-0.943130,-0.943130,126.839999,13.349167
4,2022-01-31,ZS,-0.049596,0.049596,0.049596,256.129498,19.769545
5,2022-01-31,MDLZ,0.413248,-0.413248,-0.413248,59.587012,0.512308
6,2022-01-31,XEL,-1.515554,1.515554,1.515554,59.304028,0.596818
7,2022-01-31,VRSN,0.703379,-0.703379,-0.703379,223.682178,13.858297
8,2022-01-31,INSM,0.452879,-0.452879,-0.452879,23.700000,2.252257
9,2022-01-31,ON,0.431625,-0.431625,-0.431625,61.594499,6.011007


## 4. Filtered/ranked/target-weight table

In [11]:
df_ranked.head(16)

,date,ticker,score,z_score,raw_z_score,window_mean,window_std,filtered_out,rank,target_weight
0,2022-01-31,ALGN,0.466693,-0.466693,-0.466693,520.869496,55.517293,False,7.0,0.142857
1,2022-01-31,AAPL,-0.727530,0.727530,0.727530,166.062898,6.609396,False,25.0,0.000000
2,2022-01-31,PANW,-0.398864,0.398864,0.398864,84.915417,3.304162,False,23.0,0.000000
3,2022-01-31,AMD,0.943130,-0.943130,-0.943130,126.839999,13.349167,False,1.0,0.142857
4,2022-01-31,ZS,-0.049596,0.049596,0.049596,256.129498,19.769545,False,21.0,0.000000
5,2022-01-31,MDLZ,0.413248,-0.413248,-0.413248,59.587012,0.512308,False,11.0,0.000000
6,2022-01-31,XEL,-1.515554,1.515554,1.515554,59.304028,0.596818,False,28.0,0.000000
7,2022-01-31,VRSN,0.703379,-0.703379,-0.703379,223.682178,13.858297,False,4.0,0.142857
8,2022-01-31,INSM,0.452879,-0.452879,-0.452879,23.700000,2.252257,False,9.0,0.000000
9,2022-01-31,ON,0.431625,-0.431625,-0.431625,61.594499,6.011007,False,10.0,0.000000


## 5. Cross-check: vectorized score vs the existing per-ticker `MeanReversionScorer`

Picks one (date, ticker) pair, replays the same window slice through the
class-based scorer, and confirms the two numbers agree within tolerance.

In [12]:
check_date = rebalance_dates[10]
check_ticker = tickers[3]

vectorized_score = df_metrics.loc[
    (df_metrics["date"] == check_date) & (df_metrics["ticker"] == check_ticker), "score"
].iloc[0]

history = price_df.loc[:check_date, check_ticker].dropna()
window_slice = history.iloc[-WINDOW:] if len(history) >= WINDOW else history
classic_score, classic_metrics = MeanReversionScorer(min_periods=20, clip_z=3.0).compute_score(window_slice.values)

print(f"vectorized: {vectorized_score:.6f}")
print(f"classic:    {classic_score:.6f}")
print(f"match: {abs(vectorized_score - classic_score) < 1e-6}")

vectorized: -1.221381
classic:    -1.221381
match: True


In [13]:
vectorized_score

np.float64(-1.2213814046017721)

## 6. Liquidity filter (vectorized equivalent of src_old/filters/liquidfilter.py)


In [14]:
volume_df = universe_data["volume"]

liquidity_filter = vector_calc.make_liquidity_filter(
    price_df, volume_df, min_dollar_volume=50_000_000.0, volume_window=20,
)

df_metrics_liq, df_ranked_liq = vector_calc.build_target_weight_table(
    price_df,
    rebalance_dates,
    score_fn=vector_calc.rolling_mean_reversion_score,
    window=WINDOW,
    top_percent=0.25,
    allocation_type="equal",
    filter_fn=liquidity_filter,
)

print(f"filtered out (null_filter):     {df_ranked['filtered_out'].sum()} / {len(df_ranked)}")
print(f"filtered out (liquidity_filter): {df_ranked_liq['filtered_out'].sum()} / {len(df_ranked_liq)}")
df_ranked_liq[df_ranked_liq['filtered_out']].head(10)


filtered out (null_filter):     0 / 1479
filtered out (liquidity_filter): 27 / 1479


,date,ticker,score,z_score,raw_z_score,window_mean,window_std,filtered_out,rank,target_weight
8,2022-01-31,INSM,0.452879,-0.452879,-0.452879,23.700000,2.252257,True,NaN,0.0
16,2022-01-31,CEG,-2.584735,2.584735,2.584735,41.573277,1.819704,True,NaN,0.0
37,2022-02-28,INSM,-0.301109,0.301109,0.301109,23.386410,1.705658,True,NaN,0.0
66,2022-03-31,INSM,-0.368318,0.368318,0.368318,23.074667,1.154799,True,NaN,0.0
95,2022-04-29,INSM,1.352562,-1.352562,-1.352562,23.366500,1.032485,True,NaN,0.0
124,2022-05-31,INSM,1.503915,-1.503915,-1.503915,22.206500,2.251789,True,NaN,0.0
153,2022-06-30,INSM,0.436898,-0.436898,-0.436898,20.830000,2.540640,True,NaN,0.0
182,2022-07-29,INSM,-0.796406,0.796406,0.796406,20.472667,2.068460,True,NaN,0.0
211,2022-08-31,INSM,-0.601202,0.601202,0.601202,22.821167,2.992062,True,NaN,0.0
240,2022-09-30,INSM,1.337504,-1.337504,-1.337504,24.116667,1.926473,True,NaN,0.0


## 7. Full backtest using the precomputed target_weight table

In [15]:
g_state, audit_ledger = iteration.run_daily_iteration(
    df_ranked,
    world_data_dict,
    rebalance_dates,
    initial_capital=100_000.0,
)

df_nav = g_state.export_nav_dataframe()
df_nav.tail()


,Core_NAV,Tactical_NAV,Total_NAV
Date,,,
2026-03-25,134362.191006,0.0,134362.191006
2026-03-26,134397.659878,0.0,134397.659878
2026-03-27,129264.610856,0.0,129264.610856
2026-03-30,132319.342017,0.0,132319.342017
2026-03-31,135451.545279,0.0,135451.545279


In [16]:
evaluation.calculate_metrics(df_nav[['Total_NAV']])


,CAGR,Sharpe_Ratio,Sortino_Ratio,Max_Drawdown,Calmar_Ratio
Total_NAV,0.074223,0.480603,0.834211,0.248109,0.299155
